# 01 · Explore the raw data


## Setup

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# print("working dir:", Path.cwd())

In [3]:
RAW = Path("../data/raw")
print("data/raw exists?", RAW.exists())

csv_files = sorted(RAW.glob("*.csv"))   # find all .csv files, sorted by name

print("files found:", len(csv_files))
for f in csv_files:
    print(" -", f.name)

data/raw exists? True
files found: 7
 - census_county_population.csv
 - complications_and_deaths.csv
 - footnote_crosswalk.csv
 - hcahps_patient_survey.csv
 - hospital_general_information.csv
 - ruca_zip_2010.csv
 - unplanned_hospital_visits.csv


## Hospital General Information

One row per hospital. The spine of the scorecard.

### Load the file

In [4]:
hospital_info = pd.read_csv(
    RAW / "hospital_general_information.csv",
    dtype=str, # only the zip code is the nonstring object
    keep_default_na=False,
)
hospital_info.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5419 entries, 0 to 5418
Data columns (total 38 columns):
 #   Column                                            Non-Null Count  Dtype 
---  ------                                            --------------  ----- 
 0   Facility ID                                       5419 non-null   object
 1   Facility Name                                     5419 non-null   object
 2   Address                                           5419 non-null   object
 3   City/Town                                         5419 non-null   object
 4   State                                             5419 non-null   object
 5   ZIP Code                                          5419 non-null   object
 6   County/Parish                                     5419 non-null   object
 7   Telephone Number                                  5419 non-null   object
 8   Hospital Type                                     5419 non-null   object
 9   Hospital Ownership            

### Filter to New Mexico

In [5]:
print(hospital_info['State'].value_counts())

State
TX    467
CA    377
FL    221
OH    193
IL    193
NY    192
PA    186
LA    161
GA    149
MI    149
IN    148
WI    140
KS    139
OK    136
MN    136
TN    121
MO    120
NC    120
IA    118
MS    109
AZ    108
KY    103
WA    100
AL     99
VA     96
CO     96
NE     93
AR     90
MA     84
NJ     78
SC     66
MT     63
OR     62
SD     61
PR     59
MD     56
WV     55
UT     52
ID     48
ND     47
NV     46
NM     45
ME     36
CT     36
WY     31
NH     27
AK     25
HI     23
VT     17
RI     13
DE     13
DC     10
VI      2
GU      2
AS      1
MP      1
Name: count, dtype: int64


In [6]:
hospital_info_nm = hospital_info[hospital_info["State"] == "NM"].copy()

### Look at the key columns

In [7]:
print(hospital_info_nm['Hospital Type'].value_counts())

Hospital Type
Acute Care Hospitals                    26
Critical Access Hospitals               13
Psychiatric                              4
Acute Care - Veterans Administration     1
Rural Emergency Hospital                 1
Name: count, dtype: int64


In [8]:
print(hospital_info_nm['Hospital Ownership'].value_counts())


Hospital Ownership
Voluntary non-profit - Private                 15
Proprietary                                    15
Government - Federal                            5
Government - Local                              4
Government - State                              2
Government - Hospital District or Authority     2
Veterans Health Administration                  1
Voluntary non-profit - Other                    1
Name: count, dtype: int64


In [9]:
print(hospital_info_nm['Hospital overall rating'].value_counts())

Hospital overall rating
Not Available    22
2                11
3                 5
4                 3
1                 3
5                 1
Name: count, dtype: int64


In [10]:
print(hospital_info_nm['Hospital overall rating footnote'].value_counts())

Hospital overall rating footnote
      23
16    17
19     5
Name: count, dtype: int64


- 16 → "There are too few measures or measure groups reported to calculate a star rating..."
- 19 → "Data are shown only for hospitals that participate in the IQR and OQR programs"

### Takeaway — Hospital General Information

- **Grain:** one row per hospital. 45 NM hospitals, 38 columns.
- **Types:** 
  - 26 Acute Care, 
  - 13 Critical Access, 
  - 4 Psychiatric, 1 Acute Care(VA), 
  - 1 Rural Emergency. 

The 13 Critical Access hospitals are pretty important: they are often exempt from several CMS payment-linked quality program, and their low patient volumes often leave them without reportable quality score.

- **Ownership:** an even split between 
  - non-profit private (15)
  - proprietary/for-profit (15)
  - the rest are government or VA.
- **Overall star rating:** only **23 of 45** hospitals have a rating; **22 are "Not Available."**
  - 17 missing due to footnote **16** ("too few measures to calculate a star rating")
  - 5 missing due to footnote **19** (shown only for hospitals in the IQR/OQR reporting programs)
- **Role:** this is the spine. Every other file joins onto it by Facility ID.
- **Question-2 signal:** nearly half of NM hospitals have no overall rating. To check later: are the 22 unrated mostly Critical Access / rural?


Notes for myself:

What does the CMS (Center for Medicare & Medicaid Services) do?

They are the main data source and measurement authority.
Collecting data such as:
- medicare claims
- hospital submissions
- patient surverys
and then taking the data and calculates standardized quality measures so hospitals can be compared more fairly. 


The term crticial in Critical Acess is more about how criticial that facility is to the community it is in

## Unplanned Hospital Visits

### Loading the file

In [11]:
unplanned_visits = pd.read_csv(
    RAW / "unplanned_hospital_visits.csv",
    dtype=str, # only the zi|p code is the nonstring object
    keep_default_na=False,
)
unplanned_visits.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67060 entries, 0 to 67059
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Facility ID                  67060 non-null  object
 1   Facility Name                67060 non-null  object
 2   Address                      67060 non-null  object
 3   City/Town                    67060 non-null  object
 4   State                        67060 non-null  object
 5   ZIP Code                     67060 non-null  object
 6   County/Parish                67060 non-null  object
 7   Telephone Number             67060 non-null  object
 8   Measure ID                   67060 non-null  object
 9   Measure Name                 67060 non-null  object
 10  Compared to National         67060 non-null  object
 11  Denominator                  67060 non-null  object
 12  Score                        67060 non-null  object
 13  Lower Estimate               67

In [12]:
print(unplanned_visits['State'].value_counts())

State
TX    5684
CA    4718
FL    2674
IL    2478
OH    2296
NY    2282
PA    2240
GA    1904
KS    1876
MI    1834
WI    1778
MN    1764
OK    1736
IN    1638
LA    1610
IA    1596
MO    1498
NC    1484
TN    1414
MS    1386
KY    1302
WA    1274
NE    1274
AL    1260
CO    1232
VA    1218
AZ    1218
AR    1120
MA     910
NJ     896
MT     868
OR     840
SD     840
SC     812
PR     770
WV     700
UT     672
MD     672
ND     630
ID     616
NM     574
NV     532
ME     448
WY     420
CT     392
NH     364
AK     322
HI     322
VT     210
RI     154
DC     112
DE     112
VI      28
GU      28
AS      14
MP      14
Name: count, dtype: int64


Filtering the unplanned hospital visits to New Mexico

In [13]:
unplanned_visits_nm = unplanned_visits[unplanned_visits["State"] == "NM"].copy()

In [14]:
unplanned_visits_nm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 574 entries, 38948 to 39521
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Facility ID                  574 non-null    object
 1   Facility Name                574 non-null    object
 2   Address                      574 non-null    object
 3   City/Town                    574 non-null    object
 4   State                        574 non-null    object
 5   ZIP Code                     574 non-null    object
 6   County/Parish                574 non-null    object
 7   Telephone Number             574 non-null    object
 8   Measure ID                   574 non-null    object
 9   Measure Name                 574 non-null    object
 10  Compared to National         574 non-null    object
 11  Denominator                  574 non-null    object
 12  Score                        574 non-null    object
 13  Lower Estimate               574 n

In [15]:
print("unique hospitals: ", unplanned_visits_nm["Facility Name"].nunique())
print("----")
print(unplanned_visits_nm["Facility Name"].value_counts())

unique hospitals:  41
----
Facility Name
UNM HOSPITAL                                        14
LOVELACE WESTSIDE HOSPITAL                          14
MOUNTAIN VIEW REGIONAL MEDICAL CENTER               14
LOVELACE REGIONAL HOSPITAL - ROSWELL                14
PRESBYTERIAN SANTA FE MEDICAL CENTER                14
THREE CROSSES REGIONAL HOSPITAL LLC                 14
GUADALUPE COUNTY HOSPITAL                           14
SIERRA VISTA HOSPITAL                               14
SOCORRO GENERAL HOSPITAL                            14
DR DAN C TRIGG MEMORIAL HOSPITAL                    14
UNION COUNTY GENERAL HOSPITAL                       14
NOR-LEA HOSPITAL DISTRICT                           14
LINCOLN COUNTY MEDICAL CENTER                       14
MINERS' COLFAX MEDICAL CENTER                       14
CIBOLA GENERAL HOSPITAL                             14
MIMBRES VALLEY MEDICAL CENTER                       14
HOLY CROSS HOSPITAL A DIV OF TAOS HEALTH SYSTEMS    14
GILA REGIONAL MEDICAL CE

In [16]:
print(unplanned_visits_nm["Measure Name"].value_counts())

Measure Name
Hospital return days for heart attack patients                                             41
Hospital return days for heart failure patients                                            41
Hospital return days for pneumonia patients                                                41
Hybrid Hospital-Wide All-Cause Readmission Measure (HWR)                                   41
Rate of unplanned hospital visits after colonoscopy (per 1,000 colonoscopies)              41
Rate of inpatient admissions for patients receiving outpatient chemotherapy                41
Rate of emergency department (ED) visits for patients receiving outpatient chemotherapy    41
Ratio of unplanned hospital visits after hospital outpatient surgery                       41
Acute Myocardial Infarction (AMI) 30-Day Readmission Rate                                  41
Rate of readmission for CABG                                                               41
Rate of readmission for chronic obstructive pul

This is a bit odd because that means that these columns only appear at 41 hospitals, when we saw that there was 45 hospitals in New Mexico in the prev dataset

In [17]:
print(unplanned_visits_nm["Compared to National"].value_counts())

Compared to National
Not Available                          468
No Different Than the National Rate     43
Number of Cases Too Small               35
No Different than expected              19
Number of cases too small                7
Better than expected                     1
Worse than expected                      1
Name: count, dtype: int64


In [18]:
unplanned_visits_nm["Score"].value_counts()

Score
Not Available    338
1                 10
12.8               5
5.9                4
1.1                4
                ... 
-46.8              1
16.4               1
22.2               1
-34.3              1
12.6               1
Name: count, Length: 165, dtype: int64

In [19]:
display(
    unplanned_visits_nm[
        ["Facility Name", "Measure Name", "Number of Patients", "Denominator", "Score"]
    ].head(20)
)

,Facility Name,Measure Name,Number of Patients,Denominator,Score
38948,UNM HOSPITAL,Hospital return days for heart attack patients,78,80,-1.9
38949,UNM HOSPITAL,Hospital return days for heart failure patients,170,204,8.7
38950,UNM HOSPITAL,Hospital return days for pneumonia patients,204,221,5.5
38951,UNM HOSPITAL,Hybrid Hospital-Wide All-Cause Readmission Mea...,Not Applicable,Not Available,Not Available
38952,UNM HOSPITAL,Rate of unplanned hospital visits after colono...,Not Applicable,930,11.4
38953,UNM HOSPITAL,Rate of inpatient admissions for patients rece...,Not Applicable,410,9
38954,UNM HOSPITAL,Rate of emergency department (ED) visits for p...,Not Applicable,410,5.4
38955,UNM HOSPITAL,Ratio of unplanned hospital visits after hospi...,Not Applicable,717,0.9
38956,UNM HOSPITAL,Acute Myocardial Infarction (AMI) 30-Day Readm...,Not Applicable,155,14.1
38957,UNM HOSPITAL,Rate of readmission for CABG,Not Applicable,35,11.7


### Takeaway — Unplanned Hospital Visits (readmissions)

- **Grain:** one row per hospital-measure. 574 NM rows = 41 hospitals × 14 measures.
- **Coverage:** only **41 of 45** NM hospitals appear here 
- **Measures:** 14 different measures for each hospital.
- **Scores:** 236 of 574 hospital-measures have a value; 338 are missing.
- **Missingness is large:** **468 of 574 rows (82%)** are "Not Available" in Compared to National. Only 2 rows are flagged Better/Worse (1 each); the rest are "No Different" or "Too Few."
- **Clean-up notes for 02_clean:**
  - Compared to National uses two vocabularies: 
    - "…than the National Rate" (rate measures) vs "…than expected" (EDAC day measures).
  - Can combine the 2 categories that are spelled differently:
    - "Number of Cases Too Small"
    - "Number of cases too small"
- **Question-2 signal:** most NM hospital-measures can't be scored, so the readmission scorecard will cover only a handful of hospitals (338/574 do not have scores)


## Complications and Deaths

In [20]:
complications_death = pd.read_csv(
    RAW / "complications_and_deaths.csv",
    dtype=str, # only the zip code is the nonstring object
    keep_default_na=False,
)

In [21]:
complications_death_nm = complications_death[complications_death["State"] == "NM"].copy()
complications_death_nm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 820 entries, 55640 to 56459
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Facility ID           820 non-null    object
 1   Facility Name         820 non-null    object
 2   Address               820 non-null    object
 3   City/Town             820 non-null    object
 4   State                 820 non-null    object
 5   ZIP Code              820 non-null    object
 6   County/Parish         820 non-null    object
 7   Telephone Number      820 non-null    object
 8   Measure ID            820 non-null    object
 9   Measure Name          820 non-null    object
 10  Compared to National  820 non-null    object
 11  Denominator           820 non-null    object
 12  Score                 820 non-null    object
 13  Lower Estimate        820 non-null    object
 14  Higher Estimate       820 non-null    object
 15  Footnote              820 non-null    o

In [22]:
print(complications_death_nm["Facility Name"].value_counts())

Facility Name
UNM HOSPITAL                                        20
LOVELACE WESTSIDE HOSPITAL                          20
MOUNTAIN VIEW REGIONAL MEDICAL CENTER               20
LOVELACE REGIONAL HOSPITAL - ROSWELL                20
PRESBYTERIAN SANTA FE MEDICAL CENTER                20
THREE CROSSES REGIONAL HOSPITAL LLC                 20
GUADALUPE COUNTY HOSPITAL                           20
SIERRA VISTA HOSPITAL                               20
SOCORRO GENERAL HOSPITAL                            20
DR DAN C TRIGG MEMORIAL HOSPITAL                    20
UNION COUNTY GENERAL HOSPITAL                       20
NOR-LEA HOSPITAL DISTRICT                           20
LINCOLN COUNTY MEDICAL CENTER                       20
MINERS' COLFAX MEDICAL CENTER                       20
CIBOLA GENERAL HOSPITAL                             20
MIMBRES VALLEY MEDICAL CENTER                       20
HOLY CROSS HOSPITAL A DIV OF TAOS HEALTH SYSTEMS    20
GILA REGIONAL MEDICAL CENTER                       

In [23]:
print(complications_death_nm["Compared to National"].value_counts())

Compared to National
Not Available                           541
No Different Than the National Rate     207
Number of Cases Too Small                40
No Different Than the National Value     19
Worse Than the National Rate              8
Worse Than the National Value             5
Name: count, dtype: int64


In [24]:
# How many hospitals actually get a score, vs. one that CMS suppressed?
# Score is text here ("Not Available"/"Not Applicable"), so convert to numbers first.
score = pd.to_numeric(complications_death_nm["Score"], errors="coerce")
print("Scored rows:", score.notna().sum())
print("Missing score:", score.isna().sum())


Scored rows: 388
Missing score: 432


In [25]:
print(complications_death_nm["Measure Name"].value_counts())

Measure Name
Rate of complications for hip/knee replacement patients                      41
Hybrid Hospital-Wide All-Cause Risk Standardized Mortality Rate              41
Abdominopelvic accidental puncture or laceration rate                        41
Postoperative wound dehiscence rate                                          41
Postoperative sepsis rate                                                    41
Perioperative pulmonary embolism or deep vein thrombosis rate                41
Postoperative respiratory failure rate                                       41
Postoperative acute kidney injury requiring dialysis rate                    41
Postoperative hemorrhage or hematoma rate                                    41
In-hospital fall-associated fracture rate                                    41
Iatrogenic pneumothorax rate                                                 41
Death rate among surgical inpatients with serious treatable complications    41
Pressure ulcer rate        

### Takeaway — Complications and Death

- **Grain:** one row per complication. 820 NM rows = 41 hospitals × 20 measures.
- **Coverage:** only **41 of 45** NM hospitals appear here 
- **Measures:** 20 different measures for each hospital.
- **Missingness is large:** **541 of 820 rows (66%)** are "Not Available" in Compared to National.
- **Scores:** **388 of 820 rows have a score; 432 are missing** (from the score-count cell above).
- **Clean-up notes for 02_clean:**
  - Compared to National uses two vocabularies: 
    - "Worse Than the National Rate " vs "Worse Than the National Value "

## Patient Survey (HCAHPS)

In [26]:
hcahps = pd.read_csv(
    RAW / "hcahps_patient_survey.csv",
    dtype=str, 
    keep_default_na=False,
)
hcahps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325720 entries, 0 to 325719
Data columns (total 22 columns):
 #   Column                                 Non-Null Count   Dtype 
---  ------                                 --------------   ----- 
 0   Facility ID                            325720 non-null  object
 1   Facility Name                          325720 non-null  object
 2   Address                                325720 non-null  object
 3   City/Town                              325720 non-null  object
 4   State                                  325720 non-null  object
 5   ZIP Code                               325720 non-null  object
 6   County/Parish                          325720 non-null  object
 7   Telephone Number                       325720 non-null  object
 8   HCAHPS Measure ID                      325720 non-null  object
 9   HCAHPS Question                        325720 non-null  object
 10  HCAHPS Answer Description              325720 non-null  object
 11  

In [27]:
hcahps_nm = hcahps[hcahps["State"] == "NM"].copy()
hcahps_nm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2788 entries, 189176 to 191963
Data columns (total 22 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   Facility ID                            2788 non-null   object
 1   Facility Name                          2788 non-null   object
 2   Address                                2788 non-null   object
 3   City/Town                              2788 non-null   object
 4   State                                  2788 non-null   object
 5   ZIP Code                               2788 non-null   object
 6   County/Parish                          2788 non-null   object
 7   Telephone Number                       2788 non-null   object
 8   HCAHPS Measure ID                      2788 non-null   object
 9   HCAHPS Question                        2788 non-null   object
 10  HCAHPS Answer Description              2788 non-null   object
 11  Patient Survey 

In [28]:
print("unique hospitals:", hcahps_nm["Facility ID"].nunique())
print("unique measures:", hcahps_nm["HCAHPS Measure ID"].nunique())

unique hospitals: 41
unique measures: 68


In [29]:
hcahps_nm[["HCAHPS Measure ID", "HCAHPS Question"]].drop_duplicates()

,HCAHPS Measure ID,HCAHPS Question
189176,H_COMP_1_A_P,"Patients who reported that their nurses ""Alway..."
189177,H_COMP_1_SN_P,"Patients who reported that their nurses ""Somet..."
189178,H_COMP_1_U_P,"Patients who reported that their nurses ""Usual..."
189179,H_COMP_1_LINEAR_SCORE,Nurse communication - linear mean score
189180,H_COMP_1_STAR_RATING,Nurse communication - star rating
...,...,...
189239,H_RECMND_DY,"Patients who reported YES, they would definite..."
189240,H_RECMND_PY,"Patients who reported YES, they would probably..."
189241,H_RECMND_LINEAR_SCORE,Recommend hospital - linear mean score
189242,H_RECMND_STAR_RATING,Recommend hospital - star rating


In [30]:
summary = hcahps_nm[
    hcahps_nm["HCAHPS Measure ID"] == "H_STAR_RATING"
]

display(
    summary[
        [
            "Facility Name",
            "Patient Survey Star Rating",
            "Number of Completed Surveys",
            "Start Date",
            "End Date",
        ]
    ].head(10)
)

,Facility Name,Patient Survey Star Rating,Number of Completed Surveys,Start Date,End Date
189243,UNM HOSPITAL,2,1884,10/01/2024,09/30/2025
189311,CHRISTUS ST VINCENT REGIONAL MEDICAL CENTER,3,466,10/01/2024,09/30/2025
189379,CHRISTUS SOUTHERN NEW MEXICO,3,367,10/01/2024,09/30/2025
189447,SAN JUAN REGIONAL MEDICAL CENTER INC,2,381,10/01/2024,09/30/2025
189515,EASTERN NEW MEXICO MEDICAL CENTER,2,357,10/01/2024,09/30/2025
189583,LOVELACE MEDICAL CENTER,3,928,10/01/2024,09/30/2025
189651,PRESBYTERIAN ESPANOLA HOSPITAL,4,249,10/01/2024,09/30/2025
189719,LOVELACE WOMEN'S HOSPITAL,2,493,10/01/2024,09/30/2025
189787,MEMORIAL MEDICAL CENTER,2,1078,10/01/2024,09/30/2025
189855,PRESBYTERIAN HOSPITAL,2,5750,10/01/2024,09/30/2025


In [31]:
# showing the ratings for EVERYTHING under HCAHPS Measure ID
print(hcahps_nm["Patient Survey Star Rating"].value_counts())

Patient Survey Star Rating
Not Applicable    2419
Not Available      135
3                   80
2                   71
4                   64
1                   11
5                    8
Name: count, dtype: int64


In [32]:
surveys = hcahps_nm["Number of Completed Surveys"]
surveys[pd.to_numeric(surveys, errors="coerce").isna()].value_counts()

Number of Completed Surveys
Not Available    544
Name: count, dtype: int64

In [33]:
# overall ratings for each hospital:
summary = hcahps_nm[hcahps_nm["HCAHPS Measure ID"] == "H_STAR_RATING"]
print("hospitals with a summary-star row:", summary["Facility ID"].nunique())
print(summary["Patient Survey Star Rating"].value_counts())


hospitals with a summary-star row: 41
Patient Survey Star Rating
Not Available    15
3                10
2                 9
4                 7
Name: count, dtype: int64


### Takeaway — Patient Survey (HCAHPS)

- **Grain:** one row per hospital per HCAHPS measure. 2,788 NM rows = 41 hospitals × 68 measures.
- **Coverage:** 41 of 45 NM hospitals (same 4 missing as the other measure files).
- **Measures:** 68 HCAHPS items per hospital 
- **Star rating:** of 2,788 rows 
  - **234** carry a 1–5 star; 
  - **135** are "Not Available"; 
  - **2,419** are "Not Applicable" 
- **Clean-up notes for 02_clean:**
  - Keep "Not Applicable" separate from "Not Available"
    - will need to reference the "Not Available" with the footnotes for context
- **Question-2 signal:** only 26/45 New Mexico hospitals have usable overall patient experience stars

## Footnote Crosswalk

In [34]:
# lookup table, not hospital data — no NM filter
footnote_crosswalk = pd.read_csv(
    RAW / "footnote_crosswalk.csv",
    dtype=str,
    keep_default_na=False,
)
footnote_crosswalk

,Footnote,Footnote Text
0,1,The number of cases/patients is too few to rep...
1,2,Data submitted were based on a sample of cases...
2,3,Results are based on a shorter time period tha...
3,4,Data suppressed by CMS for one or more quarters.
4,5,Results are not available for this reporting p...
5,6,Fewer than 100 patients completed the CAHPS su...
6,7,No cases met the criteria for this measure.
7,8,The lower limit of the confidence interval can...
8,9,No data are available from the state/territory...
9,10,Very few patients were eligible for the CAHPS ...


### Takeaway — Footnote Crosswalk

- **Not hospital data:** a 32-row lookup table, 
Footnote code → Footnote Text.
- **No NM filter:** the codes apply to every state.
- **Role:** to later see if I can combine the footnotes from other datasets to these footnotes

## RUCA (rural-urban codes)

In [35]:
ruca = pd.read_csv(
    RAW / "ruca_zip_2010.csv",
    dtype=str,
    keep_default_na=False,
)
ruca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41164 entries, 0 to 41163
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ''ZIP_CODE''  41164 non-null  object
 1   STATE         41164 non-null  object
 2   ZIP_TYPE      41164 non-null  object
 3   RUCA1         41164 non-null  object
 4   RUCA2         41164 non-null  object
dtypes: object(5)
memory usage: 1.6+ MB


In [36]:
# heads-up: the ZIP column name AND its values are wrapped in quotes
print(ruca.columns.tolist())
print("raw ZIP sample:", ruca["''ZIP_CODE''"].head(3).tolist())

["''ZIP_CODE''", 'STATE', 'ZIP_TYPE', 'RUCA1', 'RUCA2']
raw ZIP sample: ["''00001''", "''00002''", "''00003''"]


In [37]:
ruca_nm = ruca[ruca["STATE"] == "NM"].copy()
print("NM ZIPs:", len(ruca_nm))
# RUCA1: 1-3 metro, 4-6 micro, 7-9 small town, 10 rural
print(ruca_nm["RUCA1"].value_counts().sort_index())

NM ZIPs: 429
RUCA1
1      80
10    124
2      86
3       2
4      43
5      44
7      34
8      16
Name: count, dtype: int64


In [38]:
# can every NM hospital ZIP find a RUCA match?
# strip the quotes off the RUCA ZIPs first, then compare on 5-digit ZIP
ruca_zip = ruca["''ZIP_CODE''"].str.replace("'", "", regex=False).str[:5]
hzip = hospital_info_nm["ZIP Code"].str[:5]

matched = hzip.isin(set(ruca_zip))
print("NM hospital ZIPs matched to RUCA:", int(matched.sum()), "/", len(hzip))
print("unmatched:", sorted(hzip[~matched].unique()))

NM hospital ZIPs matched to RUCA: 45 / 45
unmatched: []


### Takeaway — RUCA

- **Grain:** one row per ZIP code. This is USDA geographic data, not hospital data.
- **NM coverage:** 429 ZIP codes:
  - 168 metropolitan
  - 87 micropolitan
  - 50 small town
  - 124 rural
- **Hospital coverage:** all **45 of 45** NM hospital ZIP codes match a RUCA record after cleaning.
- **Clean-up notes for 02_clean:**
  - The ZIP column is named `''ZIP_CODE''`, and its values contain extra quotes such as `''00001''`.
  - Remove the extra quotes and preserve ZIP codes as just a 5 char text
  - Join the hospital and RUCA datasets using the cleaned ZIP code.
- **Role:** adds a rural/urban classification to each hospital for the coverage analysis in question 2.
